In [ ]:
%pip install pyspark
%pip install wordcloud
%pip install sparknlp
%pip install matplotlib
%pip install seaborn

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (col, lower, trim, year, to_date, udf, from_unixtime, 
                                   from_json, expr, avg, count, corr, floor,
                                   count_distinct, min, max, when, rand, lit, abs)
from pyspark.sql.types import StructType, StructField, StringType, FloatType, ArrayType, IntegerType
from pyspark.ml.feature import StringIndexer, VectorAssembler, Word2Vec
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import seaborn as sns
from pyspark.ml.recommendation import ALS
from matplotlib.patches import Patch
from sparknlp.annotator import StopWordsCleaner, Tokenizer, Normalizer, LemmatizerModel
from pyspark.ml.feature import Normalizer as NormalizerML
from sparknlp.base import DocumentAssembler, Finisher
from pyspark.ml.feature import BucketedRandomProjectionLSH
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.linalg import DenseVector, VectorUDT
from pyspark.ml import Pipeline
from wordcloud import WordCloud
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
spark = SparkSession.builder \
    .appName("Big Data Project") \
    .config("spark.jars.packages", "com.johnsnowlabs.nlp:spark-nlp_2.12:5.5.2") \
    .config("spark.dynamicAllocation.enabled", "true") \
    .config("spark.sql.adaptive.enabled", "true") \
    .master("yarn") \
    .getOrCreate()

spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")
spark.sparkContext.setLogLevel("ERROR")

In [ ]:
# Define the schema for books information dataset
books_info_schema = StructType([
    StructField("Title", StringType(), True),
    StructField("description", StringType(), True),
    StructField("authors", StringType(), True),
    StructField("image", StringType(), True),
    StructField("previewLink", StringType(), True),
    StructField("publisher", StringType(), True),
    StructField("publishedDate", StringType(), True),
    StructField("infoLink", StringType(), True),
    StructField("categories", StringType(), True),
    StructField("ratingsCount", FloatType(), True)
])

# Load the data
books_info = spark.read.csv("hdfs:///user/username/csv_files/books_info.csv", header=True, schema=books_info_schema)

# Data preprocessing steps for books_info

# Drop the columns that are not needed
books_info = books_info.drop("previewLink", "infoLink", "review/", "publisher")

# Lowercase the description column
books_info = books_info.withColumn("description", lower(col("description")))

# Trim the columns
books_info = books_info.withColumn("Title", trim(col("Title")))
books_info = books_info.withColumn("description", trim(col("description")))

# Handle date format in the publishedDate column and extract only the year
books_info = books_info.withColumn("publishedDate", to_date(col("publishedDate"), "yyyy-MM-dd"))
books_info = books_info.withColumn("publishedDate", year(col("publishedDate")))

# Drop duplicates
books_info = books_info.dropDuplicates()

books_info = books_info.withColumn(
    "authors", from_json(col("authors"), ArrayType(StringType()))
).withColumn(
    "categories", from_json(col("categories"), ArrayType(StringType()))
)

# Drop nan values subset
books_info = books_info.dropna(subset="authors", how="any")
books_info = books_info.dropna(subset="categories", how="any")
books_info = books_info.dropna(subset="description", how="any")

# Create hash Book_Id for book Title
book_indexer = StringIndexer(inputCol="Title", outputCol="Title_index")
books_info = book_indexer.fit(books_info).transform(books_info)
books_info = books_info.withColumn("Title_index", books_info["Title_index"].cast("int"))
books_info = books_info.withColumnRenamed("Title_index","Book_Id")

# Show the books_info data
print("Books Info")
books_info.show(10)
print("Books Info Schema")
books_info.printSchema()

# Data preprocessing steps for books_rating

books_rating_schema = StructType([
    StructField("Id", StringType(), True),
    StructField("Title", StringType(), True),
    StructField("Price", FloatType(), True),
    StructField("User_id", StringType(), True),
    StructField("profileName", StringType(), True),
    StructField("review/helpfulness", StringType(), True),
    StructField("review/score", FloatType(), True),
    StructField("review/time", StringType(), True),
    StructField("review/summary", StringType(), True),
    StructField("review/text", StringType(), True)
])

# Load the data
books_rating = spark.read.csv("hdfs:///user/username/csv_files/books_rating.csv", header=True, schema=books_rating_schema)

# Data preprocessing steps for books_rating

books_rating = books_rating.withColumn(
    "review/helpfulness",
    expr("""
    CASE
        WHEN `review/helpfulness` LIKE '%/%' THEN cast(split(`review/helpfulness`, '/')[0] AS FLOAT) / cast(split(`review/helpfulness`, '/')[1] AS FLOAT)
        ELSE NULL
    END
    """)
)

# Drop unnecessary columns
books_rating = books_rating.drop("review/summary", "profileName")

books_rating = books_rating.withColumn("review/time", from_unixtime(col("review/time"), "dd-MM-yyyy"))
books_rating = books_rating.withColumn("review/time", to_date(col("review/time"), "dd-MM-yyyy"))

books_rating = books_rating.dropna(subset=["User_id"],how="any")
books_rating = books_rating.dropna(subset=["Id"],how="any")
books_rating = books_rating.dropna(subset=["review/score"],how="any")

# Drop duplicates wihthout Id
books_rating = books_rating.dropDuplicates(subset=["Title",
                                                   "Price",
                                                   "User_id",
                                                   "review/helpfulness",
                                                   "review/score",
                                                   "review/time",
                                                   "review/text"])

# Transform User_id hash to integer
user_indexer = StringIndexer(inputCol="User_id", outputCol="User_id_index")
books_rating = user_indexer.fit(books_rating).transform(books_rating)
books_rating = books_rating.withColumn("User_id_index", books_rating["User_id_index"].cast("int"))
books_rating = books_rating.drop("User_id")
books_rating = books_rating.withColumnRenamed("User_id_index", "User_id")

print("Books Rating Data")
books_rating.show(10)
print("Books Rating Schema")
books_rating.printSchema()

In [ ]:
# How many books sold each year
yearly_book_count = (
    books_rating
    .filter(col("Price").isNotNull())
    .groupBy(year("review/time").alias("year"))
    .agg(count("*").alias("book_count"))
    .orderBy("year")
)
yearly_book_count_pd = yearly_book_count.toPandas()

# Average yearly prices
avg_yearly_prices = (
    books_rating
    .filter(col("Price").isNotNull())
    .groupBy(year("review/time").alias("year"))
    .agg(avg("Price").alias("average_price"))
    .filter(col("year") >= 2000)  # focusing on 2000s
    .orderBy("year")
)
avg_yearly_prices_pd = avg_yearly_prices.toPandas()

# Distribution of book prices
books_rating_binned = books_rating.withColumn(
    "price_bin", floor(col("Price") / 5) * 5
)
price_distribution = (
    books_rating_binned
    .groupBy("price_bin")
    .count()
    .orderBy("price_bin")
)
pdf_dist = price_distribution.toPandas()
pdf_dist = pdf_dist[pdf_dist["price_bin"] <= 60]
pdf_dist["range"] = (
    pdf_dist["price_bin"].astype(int).astype(str) + "-" + (pdf_dist["price_bin"] + 5).astype(int).astype(str)
)

# Average review score vs. price range
books_rating_binned = books_rating.withColumn(
    "price_bin", floor(col("Price") / 10) * 10
)

price_score_df = (
    books_rating_binned
    .groupBy("price_bin")
    .agg(avg("review/score").alias("avg_score"))
    .orderBy("price_bin")
)
pdf_score = price_score_df.toPandas()
pdf_score = pdf_score[pdf_score["price_bin"] <= 120]
pdf_score["range"] = (
    pdf_score["price_bin"].astype(int).astype(str) + "-" + (pdf_score["price_bin"] + 10).astype(int).astype(str)
)

# Correlation between Price and Review Score
corr_val = books_rating.select(corr("Price", "review/score")).collect()[0][0]
print("Correlation between Price and Review Score:", corr_val)

# Convert the year columns to strings so the axis does not try to plot floats:
yearly_book_count_pd["year"] = yearly_book_count_pd["year"].astype(int).astype(str)
avg_yearly_prices_pd["year"] = avg_yearly_prices_pd["year"].astype(int).astype(str)

sns.set_style("whitegrid")

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Book Prices, Counts, and Reviews Analysis", fontsize=16, y=1.02)

ax1 = axes[0, 0]
sns.barplot(
    x="year", 
    y="book_count",
    data=yearly_book_count_pd, 
    palette="Blues_d",
    ax=ax1
)
ax1.set_title("Number of Books Sold Each Year", fontsize=12)
ax1.set_xlabel("Year")
ax1.set_ylabel("Count of Books")

# Make the x‐tick labels look nice
ax1.tick_params(axis='x', rotation=45)
ax1.yaxis.set_major_locator(MaxNLocator(integer=True))

ax2 = axes[0, 1]
sns.lineplot(
    x="year",
    y="average_price",
    data=avg_yearly_prices_pd,
    marker="o",
    ax=ax2,
    color="darkgreen"
)
ax2.set_title("Average Yearly Book Prices", fontsize=12)
ax2.set_xlabel("Year")
ax2.set_ylabel("Average Price ($)")
ax2.tick_params(axis='x', rotation=45)

ax3 = axes[1, 0]
sns.barplot(
    x="range",
    y="count",
    data=pdf_dist,
    palette="rocket",
    ax=ax3
)
ax3.set_title("Distribution of Book Prices", fontsize=12)
ax3.set_xlabel("Price Range ($)")
ax3.set_ylabel("Count")
ax3.tick_params(axis='x', rotation=45)
ax3.yaxis.set_major_locator(MaxNLocator(integer=True))

ax4 = axes[1, 1]
sns.barplot(
    x="range",
    y="avg_score",
    data=pdf_score,
    palette="coolwarm",
    ax=ax4
)
ax4.set_title("Average Review Score vs. Price Range", fontsize=12)
ax4.set_xlabel("Price Range ($)")
ax4.set_ylabel("Average Review Score")
ax4.tick_params(axis='x', rotation=45)
ax4.set_ylim(0, 5.2)  # ensures full range of review scores is visible

# Optional: annotate the correlation in the last subplot
ax4.text(
    0.5, 
    0.05, 
    f"Correlation(Price, Score) = {corr_val:.3f}",
    fontsize=10, 
    transform=ax4.transAxes,
    horizontalalignment='center'
)

plt.tight_layout()
plt.show()

In [ ]:
# Step 1: Group and Aggregate Data
reviews_per_year = (
    books_rating.groupBy(year("review/time").alias("year"))
    .agg(count_distinct("User_id").alias("user_count"))
    .dropna(subset=["year"])  # Drop rows where year is null
    .orderBy("year")          # Sort data by year
)

# Display Data
print("Number of Users Reviewing per Year")
reviews_per_year.show()

# Step 2: Prepare Data for Plotting
years = [row["year"] for row in reviews_per_year.collect()]
user_counts = [row["user_count"] for row in reviews_per_year.collect()]

# Step 3: Plot the Data
plt.figure(figsize=(10, 7))

sns.set_style('darkgrid')
sns.set_theme(rc={"axes.facecolor": "#F2EAC5", "figure.facecolor": "white"})

# Barplot
sns.barplot(x=years, y=user_counts, palette="viridis")
for i, count in enumerate(user_counts):
    plt.text(i, count + 0.5, str(count), ha='center', va='bottom', fontsize=10)

# Formatting the plot
plt.title("Number of Users per Year", fontsize=16, pad=20)
plt.xlabel("Year", fontsize=12, labelpad=15)
plt.ylabel("Number of Users", fontsize=12, labelpad=20)
plt.xticks(rotation=45, fontsize=10)
plt.yticks(fontsize=10)
plt.tight_layout()
plt.show()

# Step 4: Statistics
print("Statistics for the Number of Users per Year")

min_stats = reviews_per_year.agg(min("user_count").alias("min_count")).collect()[0]
max_stats = reviews_per_year.agg(max("user_count").alias("max_count")).collect()[0]
avg_stats = reviews_per_year.agg(avg("user_count").alias("avg_count")).collect()[0]

min_year = reviews_per_year.filter(col("user_count") == min_stats["min_count"]).select("year").collect()[0]["year"]
max_year = reviews_per_year.filter(col("user_count") == max_stats["max_count"]).select("year").collect()[0]["year"]

# Print the statistics
print(f"Minimum User Count: {min_stats['min_count']} in Year {min_year}")
print(f"Maximum User Count: {max_stats['max_count']} in Year {max_year}")
print(f"Average User Count: {avg_stats['avg_count']:.2f}")


In [ ]:
# Step 1: Classify Users Based on Scores
user_score_stats = books_rating.withColumn(
    "user_type",
    when(col("review/score") >= 4, "Good")
    .when(col("review/score") <= 2.5, "Bad")
    .otherwise("Neutral")
)

# Step 2: Group by User Type
user_type_count = user_score_stats.groupBy("user_type").count()

# Convert to Pandas DataFrame for Visualization
user_type_count_pd = user_type_count.toPandas()

# Step 3: Pie Chart 
plt.figure(figsize=(7, 6))
colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]
plt.pie(
    user_type_count_pd["count"],
    labels=user_type_count_pd["user_type"],
    autopct='%1.1f%%',
    startangle=90,
    colors=colors,
    textprops={"fontsize": 12}
)
plt.title("User Classification (Pie Chart)", fontsize=15, pad=20)
plt.axis("equal")
plt.tight_layout()
plt.show()

# Step 4: Calculate and Print Review Statistics
total_reviews = books_rating.count()
positive_reviews = books_rating.filter(books_rating["review/score"] >= 4).count()
negative_reviews = books_rating.filter(books_rating["review/score"] <= 2.5).count()

positive_ratio = (positive_reviews / total_reviews) * 100
negative_ratio = (negative_reviews / total_reviews) * 100

print(f"Total Reviews: {total_reviews}")
print(f"Positive Reviews: {positive_reviews} ({positive_ratio:.2f}%)")
print(f"Negative Reviews: {negative_reviews} ({negative_ratio:.2f}%)")


In [ ]:
# Step 1: Find Users' Maximum Review Scores
user_max_scores = books_rating.groupBy("User_id").agg(
    max("review/score").alias("max_score")
)

# Alias tables to avoid ambiguity
br = books_rating.alias("br")  
ums = user_max_scores.alias("ums")  

# Step 2: Find Favorite Books (Maximum Score = 5)
user_favorite_books = br.join(
    ums,
    (br["User_id"] == ums["User_id"]) & (br["review/score"] == 5),
    "inner"
).select(
    br["User_id"].alias("User_id"),
    br["Title"].alias("Title"),
    br["review/score"].alias("review_score")
)

# Step 3: Count Favorites by Book Title
popular_favorites = (
    user_favorite_books.groupBy("Title")
    .count()
    .orderBy(col("count").desc())
)

# Display the Top 10 Popular Favorite Books
popular_favorites.show(10)

# Step 4: Convert Top 10 Books to Pandas DataFrame
popular_favorites_pd = popular_favorites.limit(10).toPandas()

# Step 5: Bar Plot for Top 10 Favorite Books
plt.figure(figsize=(12, 8))
sns.set_style("whitegrid")

# Viridis renk paleti
palette = sns.color_palette("viridis", len(popular_favorites_pd))

# Barplot
sns.barplot(
    y=popular_favorites_pd["count"],
    x=range(1, len(popular_favorites_pd) + 1),
    palette=palette,
    hue=None,
    dodge=False
)

# Labeling the plot
plt.xlabel("Books (Index)", fontsize=12, labelpad=15)
plt.ylabel("Number of Favorites", fontsize=12, labelpad=15)
plt.title("Top 10 Favorite Books", fontsize=16, pad=20)

# Annotate each bar with the count value
for i, count in enumerate(popular_favorites_pd["count"]):
    plt.text(i, count + 0.5, str(count), ha='center', va='bottom', fontsize=10)

# Custom Legend with Matching Colors
book_titles = popular_favorites_pd["Title"]
handles = [
    Patch(facecolor=palette[i], edgecolor="k", label=f"{i+1}. {book_titles.iloc[i]}")
    for i in range(len(book_titles))
]

plt.legend(
    handles=handles,
    title="Book Titles",
    loc="upper center",
    bbox_to_anchor=(0.5, -0.15),
    fontsize=10,
    title_fontsize=12,
    ncol=2 
)

plt.tight_layout()
plt.show()

In [ ]:
print("Books Info")
books_info.show(10)

print("Books Rating")
books_rating.show(10)

books_rating = books_rating.join(
    books_info,
    books_rating["Title"] == books_info["Title"],
    "inner",
).select(
    books_rating["Id"],
    books_rating["User_id"],
    books_rating["review/score"],
    books_rating["review/time"],
    books_rating["Title"],
    books_rating["Price"],
    books_info["Book_Id"],
    books_rating["review/text"]
)

print("Books Rating")
books_rating.show(10)

books_rating = books_rating.dropDuplicates(subset=["Id", "User_id", "Book_Id", "review/score", "review/time", "Price", "review/text"])

In [ ]:
df = books_rating.select("User_id", "Book_Id", "review/score")
df.show()
df.count()

# ALS model initialization
# ItemCol is the book ID = Int, UserCol is the user ID, and RatingCol is the review score
model = ALS(
    userCol="User_id",
    itemCol="Book_Id",
    ratingCol="review/score",
    coldStartStrategy='drop'
)

# Split data into training and test sets
(training, test) = df.randomSplit([0.8, 0.2])
model = model.fit(training)
predictions = model.transform(test)

In [ ]:
# Select a random user from the test dataset
selected_user_id = test.select("User_id").first()["User_id"]

# Get the list of books the selected user has already reviewed
reviewed_books = (
    df.filter(col("User_id") == selected_user_id)
    .select("Book_Id")
    .rdd.flatMap(lambda x: x)
    .collect()
)

# Filter out books that the user has already reviewed
unreviewed_books_df = df.filter(~(col("Book_Id").isin(reviewed_books)))

# Select distinct books that the user has not reviewed yet
distinct_unreviewed_books = unreviewed_books_df.select("Book_Id").distinct()

# Prepare dataset for predictions by adding the selected user's ID
prediction_input_df = distinct_unreviewed_books.withColumn("User_id", lit(selected_user_id))

# Generate predictions for unreviewed books
predictions_df = model.transform(prediction_input_df)

# Get top 3 book recommendations sorted by predicted rating
top_recommendations = predictions_df.orderBy("prediction", ascending=False).limit(3)

# Join with books_info to get book details for the recommended books
recommended_books = top_recommendations.join(books_info, on="Book_Id", how="inner")

print("Recommended Books for Chosen User: ")
recommended_books.show()

# Show the chosen user's book review history
print("Chosen user's book history: ")
user_review_history = df.filter(df.User_id == selected_user_id)
user_reviewed_books = user_review_history.join(books_info, on="Book_Id", how="inner")
user_reviewed_books.show(truncate=False)


In [ ]:
evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="review/score",
    predictionCol="prediction"
)
rmse = evaluator.evaluate(predictions)
print(f"Root Mean Squared Error (RMSE): {rmse}")

# Calculate prediction errors (absolute and squared errors)
predictions = predictions.withColumn("absolute_error", abs(col("review/score") - col("prediction")))
predictions = predictions.withColumn("squared_error", pow(col("review/score") - col("prediction"), 2))

# Collect error data for visualization
error_data = predictions.select("absolute_error").rdd.flatMap(lambda x: x).collect()

# Plot the error distribution
plt.figure(figsize=(10, 6))
plt.hist(error_data, bins=30, alpha=0.7, color='blue', label="Absolute Error")
plt.title("Distribution of Absolute Prediction Errors")
plt.xlabel("Absolute Error")
plt.ylabel("Frequency")
plt.legend()
plt.show()

In [ ]:
# Content Based Filtering for Recommendations

# Getting features that will be vectorized
books_features = books_info.select("Title", "authors", "categories", "description")
books_features = books_features.drop_duplicates()

print("Books Features Data Before NLTK Pipeline")
books_features.show(10)

# NLP Pipeline

from sparknlp.base import *
from sparknlp.annotator import *

# Document Assembler
document_assembler = DocumentAssembler() \
    .setInputCol("description") \
    .setOutputCol("document")

# Tokenization
tokenizer = Tokenizer() \
    .setInputCols(["document"]) \
    .setOutputCol("token") \
    .setTargetPattern(r"[a-zA-Z0-9]+")

# Normalizer
normalizer = Normalizer() \
    .setInputCols(["token"]) \
    .setOutputCol("normalized_token") \
    .setCleanupPatterns(["[^a-zA-Z0-9\\s']", "-"])

# Stop Words Removal
stop_words_cleaner = StopWordsCleaner() \
    .setInputCols(["normalized_token"]) \
    .setOutputCol("clean_token") \
    .setCaseSensitive(False)

# Lemmatization
lemmatizer = LemmatizerModel.pretrained("lemma_antbnc") \
    .setInputCols(["clean_token"]) \
    .setOutputCol("lemma")

# Finisher
finisher = Finisher() \
    .setInputCols(["lemma"]) \
    .setOutputCols(["lemma"]) \

pipeline = Pipeline(stages=[
    document_assembler,
    tokenizer,
    normalizer,
    stop_words_cleaner,
    lemmatizer,
    finisher
])

# Fit the pipeline
nltk_model = pipeline.fit(books_features)
books_features = nltk_model.transform(books_features)

books_features = books_features.drop("document", "token", "normalized_token", "clean_token")

# Vectorization
# In this step, we will convert the text data into numerical vectors using the Word2Vec algorithm
word2vec = Word2Vec(
    vectorSize=50,   
    minCount=2,       
    maxIter=10,       
    windowSize=5,
    inputCol="lemma",
    outputCol="description_vectors"      
)

model = word2vec.fit(books_features)

# Transform the data
books_features = model.transform(books_features)

# Show the results
print("Books Features Data After Word2Vec")
books_features = books_features.drop("lemma")
books_features.show(10)

In [14]:
books_features_indexing = books_features

In [ ]:
# Encoding book features for model training

# Since the 'authors' column has many unique values, OneHotEncoding would result in too many columns.
# Instead, Word2Vec is used to generate vector representations for authors.

word2vec = Word2Vec(
    vectorSize=100,   
    minCount=1,       
    maxIter=10,       
    windowSize=5,
    inputCol="authors",
    outputCol="authors_vectors"      
)

model = word2vec.fit(books_features_indexing)
books_features_for_indexing = model.transform(books_features_indexing)

unique_categories = books_features.selectExpr("explode(categories) as categorie").distinct().collect()
categorie_list = [row["categorie"] for row in unique_categories]

# Function to perform multi-hot encoding for categories
def multi_hot_encode(categories, categorie_list):
    return [1 if categorie in categories else 0 for categorie in categorie_list]

# Define a UDF to apply multi-hot encoding transformation
multi_hot_udf = udf(lambda x: multi_hot_encode(x, categorie_list), ArrayType(IntegerType()))

# Apply multi-hot encoding to the 'categories' column
books_features_for_indexing = books_features_for_indexing.withColumn("categories_vectors", multi_hot_udf("categories"))

# Display the transformed dataset
books_features_for_indexing.show(10)

In [ ]:
# Select users who have more than one review
user_review_count = books_rating.groupBy("User_id").count()
user_review_count = user_review_count.filter(col("count") > 1).sort(col("count").desc())
user_review_count.show()

# Choose a user_id randomly
random_user_id = user_review_count.select("User_id").orderBy(rand()).limit(1).collect()[0]["User_id"]
random_user_id = 30

print("Random User ID:", random_user_id)

In [18]:
books_features = books_features_for_indexing

In [ ]:
user_books = books_rating.filter(col("User_id") == random_user_id)
user_books_features = user_books.join(books_features, "Title", "inner")
print("Choosen User Books Features")
user_books_features = user_books_features.select("User_id", "Title", "review/score", "description_vectors", "authors","categories","categories_vectors","authors_vectors","description")
user_books_features.show()

In [ ]:
total_ratings = user_books_features.select("review/score").rdd.map(lambda x: x[0]).sum()

# Compute the weighted user vector for book descriptions
user_vector_description= user_books_features.select("description_vectors", "review/score").rdd.map(
    lambda x: (DenseVector(x[0]), x[1])  
).map(
    lambda x: DenseVector([v * (x[1] / total_ratings ) for v in x[0]]) 
).reduce(
    lambda x, y: x + y  
)

# Compute the weighted user vector for book authors
user_vector_authors = user_books_features.select("authors_vectors", "review/score").rdd.map(
    lambda x: (DenseVector(x[0]), x[1])  
).map(
    lambda x: DenseVector([v * ((x[1] / total_ratings )) for v in x[0]]) 
).reduce(
    lambda x, y: x + y  
)

print("User Authors Vector (Weighted)")
print(user_vector_authors)

In [ ]:
# Compute the weighted user vector for book categories
user_vector_categories = user_books_features.select("categories_vectors", "review/score").rdd.map(
    lambda x: (DenseVector(x[0]), x[1])  
).map(
    lambda x: DenseVector([v * x[1] for v in x[0]]) 
).reduce(
    lambda x, y: x + y  
)
total_ratings = user_books_features.select("review/score").rdd.map(lambda x: x[0]).sum()
user_vector_categories = DenseVector([v / total_ratings for v in user_vector_categories])


In [ ]:
# Merge additional book information with the user's book features
user_books_features_extra_info = user_books_features.join(
    books_info,
    "Title",
    "inner"
)

user_books_features_extra_info.show()

In [ ]:
# Convert category vectors to DenseVector format for compatibility with VectorAssembler
def to_dense_vector(array):
    return DenseVector(array)

to_dense_vector_udf = udf(to_dense_vector, VectorUDT())

books_features = books_features.withColumn("categories_vectors", to_dense_vector_udf("categories_vectors"))

assembler = VectorAssembler(
    inputCols=["description_vectors", "categories_vectors", "authors_vectors"],
    outputCol="features"
)
books_features = assembler.transform(books_features)

In [24]:
user_vector_df = spark.createDataFrame([
    (random_user_id, user_vector_categories, user_vector_description, user_vector_authors)
], ["User_id", "categories_vectors", "description_vectors", "authors_vectors"])

In [ ]:
# Assemble the user's feature vectors into a single feature column
user_vector_assembler = VectorAssembler(
    inputCols=["description_vectors", "categories_vectors", "authors_vectors"],
    outputCol="features"
)

user_vector_df = user_vector_assembler.transform(user_vector_df)

In [ ]:
user_vector = user_vector_df.select("features").collect()[0]["features"]

In [ ]:
# LSH Model
brp = BucketedRandomProjectionLSH(
    inputCol="features", 
    outputCol="hashes", 
    bucketLength=3.0
)

brp_model = brp.fit(books_features)

similar_books = brp_model.approxNearestNeighbors(books_features, user_vector, numNearestNeighbors=10)

similar_books.select("Title", "distCol").show(10, truncate=False)

recommendations = similar_books.select("Title", "distCol").limit(3)

user_books_features.show()

print("Recommendations for the User")
recommendations = recommendations.join(books_info, "Title", "inner").show(truncate=False)

In [28]:
# Normalization
normalizer = NormalizerML(inputCol="features", outputCol="normalized_features")
books_features = normalizer.transform(books_features)

In [ ]:
# Function to calculate cosine similarity between two vectors
def cosine_similarity(v1, v2):
    dot_product = sum([x * y for x, y in zip(v1, v2)])
    magnitude_v1 = sum([x**2 for x in v1]) ** 0.5
    magnitude_v2 = sum([x**2 for x in v2]) ** 0.5
    if magnitude_v1 == 0 or magnitude_v2 == 0:
        return 0.0
    return float(dot_product / (magnitude_v1 * magnitude_v2))

cosine_similarity_udf = udf(lambda features: cosine_similarity(user_vector.toArray(), features.toArray()), FloatType())

similar_books = books_features.withColumn(
    "cosine_similarity", cosine_similarity_udf(col("normalized_features"))
)

# Get the top 10 most similar books
similar_books = similar_books.orderBy(col("cosine_similarity").desc())

similar_books.select("Title", "cosine_similarity").show(10, truncate=False)

recommendations = similar_books.select("Title", "cosine_similarity").limit(3)
print("Recommendations for the User")
recommendations = recommendations.join(books_info, "Title", "inner").show(truncate=False)


In [ ]:
from sparknlp.annotator import ViveknSentimentModel

books_rating.show()

document = DocumentAssembler() \
    .setInputCol("review/text") \
    .setOutputCol("document")

token = Tokenizer() \
    .setInputCols(["document"]) \
    .setOutputCol("token")

normalizer = Normalizer() \
    .setInputCols(["token"]) \
    .setOutputCol("normal")

vivekn =  ViveknSentimentModel.pretrained() \
    .setInputCols(["document", "normal"]) \
    .setOutputCol("result_sentiment")

finisher = Finisher() \
.setInputCols(["result_sentiment"]) \
.setOutputCols("final_sentiment")

pipeline = Pipeline().setStages([document, token, normalizer, vivekn, finisher])

model = pipeline.fit(books_rating)

result = model.transform(books_rating)

reviews_with_sentiment = result.select("Id", "final_sentiment", "review/text", "review/score", "Title", "User_id")

In [ ]:
print("Reviews with Sentiment")
reviews_with_sentiment.show(10)
reviews_with_sentiment.printSchema()

In [ ]:
# Calculate the number of positive and negative reviews for the sentiment analysis
positive_scores = reviews_with_sentiment.filter(col("final_sentiment").getItem(0) == "positive").select("review/score").rdd.flatMap(lambda x: x).collect()
negative_scores = reviews_with_sentiment.filter(col("final_sentiment").getItem(0) == "negative").select("review/score").rdd.flatMap(lambda x: x).collect()

positive_reviews = len(positive_scores)
negative_reviews = len(negative_scores)
sizes = [positive_reviews, negative_reviews]
labels = ["Positive", "Negative"]
colors = ["#2ca02c", "#d62728"]
explode = (0.1, 0.1)

fig, ax = plt.subplots(figsize=(8, 6), dpi=100)
wedges, texts, autotexts = ax.pie(
    sizes,
    labels=labels,
    autopct='%1.1f%%',
    explode=explode,
    colors=colors,
    shadow=True,
    textprops={'fontsize': 14}
)

for text in autotexts:
    text.set_color("white")
    text.set_fontweight("bold")

for label in texts:
    label.set_fontsize(13)

ax.set_title("Distribution of Reviews by Sentiment", fontsize=18, pad=20)
fig.patch.set_facecolor('#f9f9f9')
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 Books with Most Positive Reviews
positive_books = reviews_with_sentiment.filter(col("final_sentiment").getItem(0) == "positive") \
    .groupBy("Title") \
    .count() \
    .orderBy("count", ascending=False) \
    .limit(10)  

positive_books_pd = positive_books.toPandas()

sns.set_style('darkgrid')
sns.set(rc={"axes.facecolor":"white","figure.facecolor":"#F2EAC5"})

plt.figure(figsize=(12, 8))
sns.barplot(
    x="count", 
    y="Title", 
    data=positive_books_pd, 
    palette="viridis",
    edgecolor="black"
)

plt.xlabel("Count", fontsize=14, labelpad=15)
plt.ylabel("Book Title", fontsize=14, labelpad=15)
plt.title("Top 10 Books with Most Positive Reviews", fontsize=18, pad=20)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.tight_layout()

plt.show()

In [ ]:
from pyspark.sql.functions import explode
books_sentiment_with_categories = reviews_with_sentiment.join(
    books_info, 
    "Title", 
    "inner"
).select("categories", "final_sentiment", "Title", "Id")

print("Books Sentiment with Categories")
books_sentiment_with_categories.printSchema()
books_sentiment_with_categories = books_sentiment_with_categories.withColumn("categories", explode(col("categories")))
books_sentiment_with_categories.show(10)

In [ ]:
categories_positive = books_sentiment_with_categories.filter(col("final_sentiment").getItem(0) == "positive").groupBy("categories").count().orderBy("count", ascending=False)
categories_positive.show(10, truncate=False)

In [ ]:
# Create a word cloud for the most common genres in positive reviews
wordcloud = WordCloud(
    width=800,
    height=400,
    background_color='white', 
    colormap="tab10",          
    max_font_size=70,         
    min_font_size=10           
).generate_from_frequencies(categories_positive.rdd.collectAsMap())

plt.figure(figsize=(12, 8))
plt.imshow(wordcloud)  
plt.axis('off') 
plt.title('Genres Word Cloud', fontsize=20, fontweight='bold')  
plt.show()


In [ ]:
spark.stop()